# Vectorless RAG End-to-End Notebook

This notebook mirrors the repository implementation and walks through each stage of the **Vectorless RAG** pipeline:

1. Submit a PDF to PageIndex.
2. Inspect the document tree.
3. Run single-document retrieval + generation.
4. Run multi-document synthesis.
5. Run vision-based RAG over selected pages.

> The notebook is designed as a learning and experimentation companion to the CLI scripts in the repo.

## 0) Prerequisites

- Python 3.12+
- `PAGEINDEX_API_KEY` and `OPENAI_API_KEY` in your environment or `.env` file
- Dependencies installed (`pip install -r requirements.txt`)

If you run this notebook in a clean environment, execute the setup cell below.

In [ ]:
# Optional: install dependencies in notebook environments
# %pip install -r ../requirements.txt


In [ ]:
from pathlib import Path
import sys

# Ensure imports work when notebook is opened from Notebook/
repo_root = Path.cwd().resolve().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(f"Repo root: {repo_root}")


## 1) Configuration & Sanity Checks

The project uses shared config helpers from `config.py` to validate environment variables and construct API clients.

In [ ]:
from config import get_required_env, get_openai_api_key

pageindex_key = get_required_env("PAGEINDEX_API_KEY")
openai_key = get_openai_api_key()

print("Environment looks good.")
print(f"PAGEINDEX_API_KEY loaded: {'yes' if pageindex_key else 'no'}")
print(f"OPENAI_API_KEY loaded: {'yes' if openai_key else 'no'}")


## 2) Step 1 — Submit a PDF and Wait for Indexing

This matches `step1_submit_pdf.py`.

Set `pdf_path` to one of the provided documents in `docs/` or your own PDF.

In [ ]:
from step1_submit_pdf import submit_and_wait

pdf_path = repo_root / "docs" / "annual_report.pdf"
print(f"Using PDF: {pdf_path}")

# Uncomment to run
# doc_id = submit_and_wait(str(pdf_path))
# print("doc_id:", doc_id)


## 3) Step 2 — Inspect the Hierarchical Tree

This is useful for understanding how PageIndex structured your document and which page spans map to each section.

In [ ]:
from step2_inspect_tree import inspect_tree

# Provide a previously created doc_id
doc_id = "<replace-with-doc-id>"

# Uncomment to run
# tree = inspect_tree(doc_id)


## 4) Step 3 — Single-Document Q&A (Core RAG)

This pipeline retrieves relevant context using PageIndex reasoning and then asks GPT-4o to answer strictly from that context.

In [ ]:
from step3_retrieve_generate import run_pipeline

query = "What was revenue for 2025?"

# Uncomment to run
# answer = run_pipeline(doc_id, query)
# print(answer)


## 5) Step 4 — Multi-Document Synthesis

Compare or synthesize findings across two or more PageIndex document IDs.

In [ ]:
from step4_multi_doc import multi_doc_query

multi_doc_ids = ["<doc-id-1>", "<doc-id-2>"]
comparison_query = "How did operating margin change year over year?"

# Uncomment to run
# multi_answer = multi_doc_query(multi_doc_ids, comparison_query)
# print(multi_answer)


## 6) Step 5 — Vision RAG (Charts / Tables / Scans)

For visual-heavy questions, the pipeline:
- asks PageIndex which pages are relevant,
- renders those pages as images, and
- sends them to GPT-4o vision for analysis.

In [ ]:
from step5_vision_rag import run_vision_pipeline

vision_query = "What does the revenue chart show?"

# Uncomment to run
# vision_answer = run_vision_pipeline(doc_id, str(pdf_path), vision_query)
# print(vision_answer)


## 7) Optional: Reusable Notebook Helper Layer

This cell provides utility wrappers if you want a more interactive workflow in notebooks.

In [ ]:
def ask_document(doc_id: str, question: str) -> str:
    """Convenience wrapper for single-document Q&A."""
    return run_pipeline(doc_id, question)


def compare_documents(doc_ids: list[str], question: str) -> str:
    """Convenience wrapper for multi-document Q&A."""
    if len(doc_ids) < 2:
        raise ValueError("Please provide at least two document IDs.")
    return multi_doc_query(doc_ids, question)


def ask_visual(doc_id: str, pdf_file: str, question: str) -> str:
    """Convenience wrapper for vision-based Q&A."""
    return run_vision_pipeline(doc_id, pdf_file, question)


## 8) Troubleshooting

- **Missing API key errors**: ensure `.env` contains `PAGEINDEX_API_KEY` and `OPENAI_API_KEY`.
- **`doc_id` not found**: verify the ID from the submit step and that indexing completed.
- **Vision answers are weak**: try a more explicit query and confirm referenced pages contain the relevant chart/table.

---

You now have a notebook-first interface for the same logic shipped in the CLI scripts.